Check for duplicates

In [ ]:
import pandas as pd

# Load the Excel file
file_path = '/content/wrist_fracture_2018_01_01_to_2025_07_29_excluding_2022_01_01_to_2024_08_13.xlsx'
df = pd.read_excel(file_path)

# List the available columns
print("Available columns:")
print(df.columns.tolist())

# Look at column called 'Exam Code' - list how many items there are per unique category here.
if 'Exam Code' in df.columns:
    print("\nUnique categories and their counts in 'Exam Code' column:")
    display(df['Exam Code'].value_counts())
else:
    print("\n'Exam Code' column not found in the DataFrame.")

In [ ]:
# Display all unique categories and their counts in 'Exam Code' column
if 'Exam Code' in df.columns:
    print("All unique categories and their counts in 'Exam Code' column:")
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        print(df['Exam Code'].value_counts())
else:
    print("'Exam Code' column not found in the DataFrame.")

In [ ]:
def filter_wrist_exam_codes(df, exclude=False):
    """
    Filters the DataFrame to keep or exclude rows with 'Exam Code' starting with 'XR-WRIST' or containing 'XRWRST'.

    Args:
        df (pd.DataFrame): The input DataFrame.
        exclude (bool): If True, exclude the specified exam codes. If False, keep them.

    Returns:
        pd.DataFrame: The filtered DataFrame.
    """
    if 'Exam Code' in df.columns:
        condition = df['Exam Code'].str.contains('^XR-WRIST', na=False) | df['Exam Code'].str.contains('XRWRST', na=False)
        if exclude:
            filtered_df = df[~condition]
        else:
            filtered_df = df[condition]
        return filtered_df
    else:
        print("'Exam Code' column not found in the DataFrame.")
        return df

In [ ]:
# Example usage:
filtered_df = filter_wrist_exam_codes(df)
display(filtered_df.head()['Exam Code'])
print("\nShape of the filtered DataFrame:", filtered_df.shape)

In [ ]:
df_bad = filter_wrist_exam_codes(df,exclude=True)
display(df_bad['Exam Code'].head())
print("\nShape of the filtered DataFrame:", df_bad.shape)

# Task
Save the dataframes `filtered_df` and `df_bad` to CSV files named 'XRWRIST.csv' and 'NOT_XRWRIST.csv' respectively. Save the 'Accession Number' column from `df_bad` to a CSV file named 'bad_study_ids.csv'. Write a Python function that takes the path to 'bad_study_ids.csv', a source folder path (containing JSON or PNG files), and a destination folder name as input. This function should load the accession numbers from 'bad_study_ids.csv', create the destination folder if it doesn't exist, and then move JSON files (named as AccessionNumber.json) and PNG files (named as AccessionNumber-<somecharacters>.png) from the source folder to the destination folder if their Accession Number matches an ID in 'bad_study_ids.csv'. Finally, apply this function to a folder containing JSON files and a folder containing PNG files.

## Save dataframes to csv

### Subtask:
Save `filtered_df` to a CSV named 'XRWRIST.csv' and `df_bad` to a CSV named 'NOT_XRWRIST.csv'.


**Reasoning**:
Save the `filtered_df` and `df_bad` DataFrames to CSV files.



In [ ]:
filtered_df.to_csv('XRWRIST.csv', index=False)
df_bad.to_csv('NOT_XRWRIST.csv', index=False)

## Save accession numbers

### Subtask:
Save the 'Accession Number' column from `df_bad` to a CSV named 'bad_study_ids.csv'.


**Reasoning**:
Select the 'Accession Number' column from the `df_bad` DataFrame and save it to a CSV file named 'bad_study_ids.csv' without the index.



In [ ]:
df_bad['Accession Number'].to_csv('bad_study_ids.csv', index=False)

## Create python function for file organization

### Subtask:
Write a Python function that takes the 'bad_study_ids.csv' file path, a source folder path (for JSON or PNG files), and a destination folder name as input.


**Reasoning**:
Define the Python function as requested, including the necessary imports for file system operations.



In [ ]:
import os
import shutil
import pandas as pd

def process_study_files(bad_study_ids_path, source_folder_path, destination_folder_name):
    """
    Loads accession numbers from a CSV and moves corresponding JSON and PNG files
    from a source folder to a destination folder.

    Args:
        bad_study_ids_path (str): Path to the CSV file containing bad study IDs.
        source_folder_path (str): Path to the source folder containing JSON and PNG files.
        destination_folder_name (str): Name of the destination folder to move files to.
    """
    if not os.path.exists(bad_study_ids_path):
        print(f"Error: bad_study_ids.csv not found at {bad_study_ids_path}")
        return

    try:
        bad_study_ids_df = pd.read_csv(bad_study_ids_path)
        bad_study_ids = bad_study_ids_df['Accession Number'].astype(str).tolist()
    except Exception as e:
        print(f"Error reading {bad_study_ids_path}: {e}")
        return

    destination_folder_path = os.path.join(source_folder_path, destination_folder_name)
    if not os.path.exists(destination_folder_path):
        os.makedirs(destination_folder_path)

    if not os.path.exists(source_folder_path):
        print(f"Error: Source folder not found at {source_folder_path}")
        return

    for filename in os.listdir(source_folder_path):
        file_path = os.path.join(source_folder_path, filename)
        if os.path.isfile(file_path):
            # Check for JSON files (AccessionNumber.json)
            if filename.endswith('.json'):
                accession_number = filename.replace('.json', '')
                if accession_number in bad_study_ids:
                    shutil.move(file_path, destination_folder_path)
                    print(f"Moved JSON file: {filename}")
            # Check for PNG files (AccessionNumber-<somecharacters>.png)
            elif filename.endswith('.png'):
                # Extract Accession Number from PNG filename
                parts = filename.split('-')
                if len(parts) > 1:
                    accession_number = parts[0]
                    if accession_number in bad_study_ids:
                        shutil.move(file_path, destination_folder_path)
                        print(f"Moved PNG file: {filename}")


## Load bad study ids

### Subtask:
Inside the function, load the 'Accession Number' values from 'bad_study_ids.csv'.


## Create destination folder

### Subtask:
Inside the function, create the destination folder (e.g., 'folder_NOTWRIST' or 'folder_with_pngs_NOTWRIST') if it doesn't exist.


## Move json files

### Subtask:
Inside the function, iterate through the JSON files in the source folder. If a file's name (before the extension) matches an Accession Number in the 'bad_study_ids', move the file to the destination folder.


## Move png files

### Subtask:
Inside the function, iterate through the PNG files in the source folder. If a file's name (before the first hyphen) matches an Accession Number in the 'bad_study_ids', move the file to the destination folder.


## Apply the function

### Subtask:
Call the function for both the JSON folder and the PNG folder.


**Reasoning**:
Define placeholder paths for the source folders and call the `process_study_files` function for both JSON and PNG files.



In [ ]:
# Define placeholder paths for source folders and destination folder names
json_source_folder = '/content/json_files' # Replace with the actual path to your JSON files
png_source_folder = '/content/png_files'   # Replace with the actual path to your PNG files
destination_folder_json = 'NOTWRIST_json'
destination_folder_png = 'NOTWRIST_png'
bad_study_ids_csv_path = 'bad_study_ids.csv' # Path to the CSV created in a previous step

# Call the function for JSON files
print(f"Processing JSON files in {json_source_folder}...")
process_study_files(bad_study_ids_csv_path, json_source_folder, destination_folder_json)

# Call the function for PNG files
print(f"\nProcessing PNG files in {png_source_folder}...")
process_study_files(bad_study_ids_csv_path, png_source_folder, destination_folder_png)

## Summary:

### Data Analysis Key Findings

*   Two dataframes, `filtered_df` and `df_bad`, were successfully saved to 'XRWRIST.csv' and 'NOT\_XRWRIST.csv' respectively.
*   The 'Accession Number' column from `df_bad` was extracted and saved to 'bad\_study\_ids.csv'.
*   A Python function `process_study_files` was defined to automate the process of identifying files associated with "bad" study IDs and moving them to a specified destination folder.
*   The `process_study_files` function includes logic to read 'bad\_study\_ids.csv', create a destination folder if needed, and iterate through source files to move matching JSON and PNG files.
*   The `process_study_files` function was called for both a designated JSON source folder and a designated PNG source folder.

### Insights or Next Steps

*   The developed function provides an efficient way to organize files based on a list of identifiers.
*   Ensure that the placeholder paths for the source JSON and PNG folders are updated with the actual paths to the data for the script to execute correctly.
